# Spark Structured Streaming

Databricks free

https://dbc-0b69ef93-5247.cloud.databricks.com/editor/notebooks/812992502093529?o=7474657872577658#command/7933892078762649


https://community.databricks.com/t5/learning-events/advanced-learning-festival-15-june-06-july-2026/ev-p/156932

https://customer-academy.databricks.com/learn/courses/3959/stream-processing-and-analysis-with-apache-spark?hash=3e063f193bec2e8ca7cdf56ffbdced1657bcc087&generated_by=409740


### DataStreamReader and DataStreamWriter

Two interfaces bracket every streaming pipeline:

    • DataStreamReader: creates streaming DataFrames - accessed through spark.readStream
    • DataStreamWriter: writes streaming results - accessed through DataFrame.writeStream

In [0]:

# Create a streaming DataFrame from JSON files in a directory
#stream_df = (
#    spark.readStream
#    .format("json")
#    .schema(order_schema)              # file sources require an explicit schema
#    .option("maxFilesPerTrigger", 1)   # throttle: one file per micro-batch
#    .load("/Volumes/catalog/schema/volume/orders")
#)


# Write streaming results to a UC table
#query = (
#    stream_df.writeStream
#    .format("delta")
#    .outputMode("append")
#    .option("checkpointLocation", checkpoint_path)
#    .toTable("orders_bronze")
#)

⚠️

Schemas Are Not Inferred on File Streams

Unlike batch reads, streaming file sources require the schema up front - inferring it would mean scanning an unbounded source. Define a StructType (or use Auto Loader's schema inference, which manages this for you).


Para rodar esse exemplo de DataStreamReader e DataStreamWriter de forma autônoma em um notebook Databricks (incluindo Serverless), o código abaixo cria os caminhos temporários, gera arquivos JSON simulados, define o schema explícito (StructType), processa o stream e consulta a tabela resultante.

## Configuração do ambiente e geração de dados JSON de teste

Cria um diretório de entrada, o caminho de checkpoint e grava dois arquivos JSON para simular a chegada incremental de pedidos.

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS main.default.streaming_demo")
base_path = "/Volumes/main/default/streaming_demo"
dbutils.fs.mkdirs(f"{base_path}/orders")
dbutils.fs.mkdirs(f"{base_path}/checkpoints/orders_bronze")
print("Volume e diretórios criados:", dbutils.fs.ls(base_path))

Volume e diretórios criados: [FileInfo(path='dbfs:/Volumes/main/default/streaming_demo/checkpoints/', name='checkpoints/', size=0, modificationTime=1795210568835), FileInfo(path='dbfs:/Volumes/main/default/streaming_demo/orders/', name='orders/', size=0, modificationTime=1795210568835)]


In [0]:
import json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType

# 1. Definir caminhos (ajuste para seu Catalog/Schema/Volume se preferir, ou use /tmp/dbfs se disponível)
# Exemplo usando Unity Catalog Volume (recomendado no Serverless):
# Certifique-se de que o volume exista ou altere 'main.default.streaming_demo' para um volume seu
table_name = "main.default.orders_bronze"

In [0]:


# Corrige os caminhos: o catalog 'main' não existe neste workspace — usar o catalog 'workspace'
base_path = "/Volumes/workspace/default/streaming_demo"
input_path = f"{base_path}/orders"
checkpoint_path = f"{base_path}/checkpoints/orders_bronze"
#table_name = "workspace.default.orders_bronze"
table_name = "workspace.default.streaming_demo"
# Cria os diretórios no Volume (caso não existam)
dbutils.fs.mkdirs(input_path)
dbutils.fs.mkdirs(checkpoint_path)

# 2. Gerar dois arquivos JSON iniciais para simular o stream
batch_1 = [
    {"order_id": "ORD-101", "customer_id": "CUST-01", "amount": 150.50, "items": 2, "event_time": "2026-11-20T18:00:00Z"},
    {"order_id": "ORD-102", "customer_id": "CUST-02", "amount": 89.90,  "items": 1, "event_time": "2026-11-20T18:01:00Z"}
]

batch_2 = [
    {"order_id": "ORD-103", "customer_id": "CUST-03", "amount": 320.00, "items": 4, "event_time": "2026-11-20T18:02:00Z"},
    {"order_id": "ORD-104", "customer_id": "CUST-01", "amount": 45.00,  "items": 1, "event_time": "2026-11-20T18:03:00Z"}
]

dbutils.fs.put(f"{input_path}/orders_01.json", "\n".join(json.dumps(r) for r in batch_1), overwrite=True)
dbutils.fs.put(f"{input_path}/orders_02.json", "\n".join(json.dumps(r) for r in batch_2), overwrite=True)

print(f"Arquivos JSON criados em: {input_path}")

Wrote 232 bytes.
Wrote 232 bytes.
Arquivos JSON criados em: /Volumes/workspace/default/streaming_demo/orders


## DataStreamReader + DataStreamWriter

Define o StructType obrigatório para fontes de arquivo, lê os JSONs com spark.readStream e grava na tabela Delta com writeStream.

In [0]:
# 1. Definir explicitamente o schema (fontes de arquivo em streaming não inferem schema)
order_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("items", IntegerType(), True),
    StructField("event_time", TimestampType(), True)
])

# 2. DataStreamReader: cria o DataFrame de streaming a partir dos arquivos JSON
stream_df = (
    spark.readStream
    .format("json")
    .schema(order_schema)               # Obrigatório para file sources
    .option("maxFilesPerTrigger", 1)    # Throttling: processa 1 arquivo por micro-batch
    .load(input_path)
)

# 3. DataStreamWriter: grava os resultados na tabela Delta do Unity Catalog
query = (
    stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)         # Processa todos os arquivos disponíveis em micro-batches (1 por vez) e para
    .toTable(table_name)
)

# Aguarda o processamento dos 2 micro-batches concluir
query.awaitTermination()
print("Streaming concluído com sucesso!")

Streaming concluído com sucesso!


Nota sobre o .trigger(availableNow=True): Como definimos .option("maxFilesPerTrigger", 1) e criamos 2 arquivos JSON, o availableNow=True executará exatamente 2 micro-batches (1 arquivo em cada) e encerrará a query automaticamente — ideal para testes em notebook sem deixar um cluster/query rodando infinitamente. Se quiser deixá-lo contínuo esperando novos arquivos, basta remover a linha .trigger(availableNow=True).

## Validar os dados gravados e o progresso da query

Verifica se os registros foram gravados na tabela Delta e inspeciona os micro-batches executados.

In [0]:
# Verificar os registros na tabela de destino
display(spark.table(table_name))

# Opcional: inspecionar os detalhes dos últimos micro-batches executados
for batch in query.recentProgress:
    print(f"Batch ID: {batch['batchId']} | Linhas lidas: {batch['numInputRows']}")

order_id,customer_id,amount,items,event_time
ORD-103,CUST-03,320.0,4,2026-11-20T18:02:00.000Z
ORD-104,CUST-01,45.0,1,2026-11-20T18:03:00.000Z
ORD-101,CUST-01,150.5,2,2026-11-20T18:00:00.000Z
ORD-102,CUST-02,89.9,1,2026-11-20T18:01:00.000Z


Batch ID: 0 | Linhas lidas: None
Batch ID: 1 | Linhas lidas: None


### Limpeza (opcional)

In [0]:
#spark.sql(f"DROP TABLE IF EXISTS {table_name}")
#dbutils.fs.rm(base_path, recurse=True)
#print("Ambiente de teste limpo.")

###  Streaming Transformations and Queries

Between the reader and the writer sits ordinary DataFrame code:

In [0]:
# Transformations: identical to batch DataFrame code, still lazy
#from pyspark.sql.functions import col


#large_orders_df = (
#    stream_df
#    .filter(col("quantity") >= 24)
#    .withColumn("is_bulk", col("quantity") >= 48)
#)


# The streaming query: starts execution and runs until stopped
#query = (
#    large_orders_df.writeStream
#    .outputMode("append")
#    .format("delta")
#    .queryName("large_orders")
#    .trigger(processingTime="1 second")
#    .option("checkpointLocation", checkpoint_path)
#    .toTable("large_orders")
#)


# Manage the query lifecycle
#query.stop()               # stop on request
#query.awaitTermination()   # block until the query terminates


    • Transformations on a streaming DataFrame produce new streaming DataFrames - nothing executes until a query starts
    • start() (or toTable()) launches a StreamingQuery that runs indefinitely until stopped or failed
    • queryName() labels the query in the Spark UI and names the temp view for memory sinks


Streaming Transformations and Queries.

Transformações lazy com filter e withColumn sobre um DataFrame de streaming (a mesma sintaxe de batch).

Configuração de .queryName(...) e .trigger(processingTime="1 second") no writeStream.

Gerenciamento do ciclo de vida da StreamingQuery (awaitTermination, isActive, status, stop()).

## Preparar dados JSON com campo quantity

Cria os arquivos de entrada simulando pedidos com diferentes quantidades (quantity) para testar o filtro (>= 24) e a coluna derivada is_bulk (>= 48).

In [0]:
import json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS main.default.streaming_demo_d")
base_path = "/Volumes/main/default/streaming_demo_d"
dbutils.fs.mkdirs(f"{base_path}/orders")
dbutils.fs.mkdirs(f"{base_path}/checkpoints/large_orders")
print("Volume e diretórios criados:", dbutils.fs.ls(base_path))

Volume e diretórios criados: [FileInfo(path='dbfs:/Volumes/main/default/streaming_demo_d/checkpoints/', name='checkpoints/', size=0, modificationTime=1795216654320), FileInfo(path='dbfs:/Volumes/main/default/streaming_demo_d/orders/', name='orders/', size=0, modificationTime=1795216654320)]


In [0]:


# Ajuste para um Catalog/Schema/Volume de sua preferência no Unity Catalog
base_path = "/Volumes/main/default/streaming_demo_d"
input_path = f"{base_path}/orders"
checkpoint_path = f"{base_path}/checkpoints/large_orders"
table_name = "main.default.large_orders"

dbutils.fs.mkdirs(input_path)
dbutils.fs.mkdirs(checkpoint_path)

batch_1 = [
    {"order_id": "ORD-201", "quantity": 10, "event_time": "2026-11-20T20:00:00Z"}, # filtrada (< 24)
    {"order_id": "ORD-202", "quantity": 24, "event_time": "2026-11-20T20:01:00Z"}, # passa (is_bulk = False)
    {"order_id": "ORD-203", "quantity": 60, "event_time": "2026-11-20T20:02:00Z"}  # passa (is_bulk = True)
]

batch_2 = [
    {"order_id": "ORD-204", "quantity": 5,  "event_time": "2026-11-20T20:03:00Z"}, # filtrada (< 24)
    {"order_id": "ORD-205", "quantity": 48, "event_time": "2026-11-20T20:04:00Z"}  # passa (is_bulk = True)
]

dbutils.fs.put(f"{input_path}/orders_01.json", "\n".join(json.dumps(r) for r in batch_1), overwrite=True)
dbutils.fs.put(f"{input_path}/orders_02.json", "\n".join(json.dumps(r) for r in batch_2), overwrite=True)

print("Dados de entrada criados com sucesso.")

Wrote 233 bytes.
Wrote 154 bytes.
Dados de entrada criados com sucesso.


# Transformações (lazy) e execução da StreamingQuery

Aplica filter e withColumn sobre o stream_df e inicia a query com queryName, processingTime e checkpointLocation.

In [0]:
from pyspark.sql.functions import col

order_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("event_time", TimestampType(), True)
])

# 1. Reader
stream_df = (
    spark.readStream
    .format("json")
    .schema(order_schema)
    .option("maxFilesPerTrigger", 1)
    .load(input_path)
)

In [0]:
#stream_df.show()

stream_df

DataFrame[order_id: string, quantity: int, event_time: timestamp]

In [0]:
#stream_df.writeStream.start()

In [0]:


# 2. Transformações: continuam lazy até o writeStream iniciar a query
large_orders_df = (
    stream_df
    .filter(col("quantity") >= 24)
    .withColumn("is_bulk", col("quantity") >= 48)
)

# 3. Query de streaming: inicia a execução
query = (
    large_orders_df.writeStream
    .outputMode("append")
    .format("delta")
    .queryName("large_orders")
    .trigger(availableNow=True)         # ProcessingTime não é suportado no compute serverless; AvailableNow processa os arquivos disponíveis e encerra
    #.trigger(processingTime="1 second")
    .option("checkpointLocation", checkpoint_path)
    .toTable(table_name)
)

print(f"Query iniciada -> Nome: {query.name} | ID: {query.id} | Ativa: {query.isActive}")

Query iniciada -> Nome: large_orders | ID: ecdb5c3c-24ad-4191-80bb-956c3f6fb9d2 | Ativa: True


# Monitorar progresso, aguardar e parar a query

Inspeciona o estado da query em execução, aguarda um curto período para processar os micro-batches e encerra a query de forma limpa com query.stop().

In [0]:
import time

# Aguarda alguns segundos para o trigger de 1s processar os arquivos disponíveis
time.sleep(5)

print("Status atual da query:", query.status)
print("Query ainda ativa antes do stop()?:", query.isActive)

# Encerra a query sob demanda
query.stop()

# Bloqueia até o encerramento completo (retorna imediatamente se já parada)
query.awaitTermination()

print("Query encerrada. Ativa agora?:", query.isActive)

Status atual da query: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}
Query ainda ativa antes do stop()?: False
Query encerrada. Ativa agora?: False


## Conferir resultado na tabela Delta e limpar ambiente

Valida apenas os pedidos com quantity >= 24 (ORD-202, ORD-203 e ORD-205) e a coluna booleana is_bulk.

In [0]:
display(spark.table(table_name).orderBy("order_id"))

# Opcional: remover tabela e arquivos temporários ao finalizar
# spark.sql(f"DROP TABLE IF EXISTS {table_name}")
# dbutils.fs.rm(base_path, recurse=True)

order_id,quantity,event_time,is_bulk
ORD-202,24,2026-11-20T20:01:00.000Z,false
ORD-203,60,2026-11-20T20:02:00.000Z,true
ORD-205,48,2026-11-20T20:04:00.000Z,true


*** Dica de ciclo de vida: Em notebooks interativos, usar query.stop() (ou trigger(availableNow=True) quando quiser drenar os dados e parar automaticamente) evita que a query continue tentando ler novos arquivos em segundo plano enquanto você edita as próximas células.

## Triggers

Triggers control when Structured Streaming processes data:


    • Default trigger (as soon as possible)

        ◦ Processes data as soon as the previous micro-batch completes

    • Fixed interval trigger (processingTime)

        ◦ Processes data at specified time intervals
        ◦ Useful for controlling resource usage

    • Available now trigger (availableNow)

        ◦ Processes all available data (in multiple batches if throttled), then stops

        ◦ Will not wait for new data to arrive afterwards - ideal for incremental batch jobs and for classroom demos

In [0]:
# Default when no trigger is specified
#df.writeStream.start()


# Process batches at fixed intervals
#df.writeStream.trigger(processingTime="2 minutes").start()


# One-time processing of available data, then stop
#df.writeStream.trigger(availableNow=True).start()



💡
A Fourth Trigger in Spark 4

On Databricks with Spark 4, the realTime trigger enables Real-Time Mode - continuous execution targeting sub-second latency. It has its own compute requirements and constraints, covered in section 4.


### E. Triggers (default, processingTime, availableNow, além de uma observação sobre realTime no Spark 4).

O código gera arquivos de entrada com throttling (maxFilesPerTrigger=1) para observar claramente como cada trigger se comporta na prática em um notebook Databricks.

### Preparar diretórios e arquivos de entrada
Cria os caminhos no Unity Catalog Volume e gera 3 arquivos JSON para que possamos observar a diferença de execução entre os triggers.

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS main.default.streaming_demo_e")
base_path = "/Volumes/main/default/streaming_demo_e"
dbutils.fs.mkdirs(f"{base_path}/events")
dbutils.fs.mkdirs(f"{base_path}/checkpoints")
print("Volume e diretórios criados:", dbutils.fs.ls(base_path))

Volume e diretórios criados: [FileInfo(path='dbfs:/Volumes/main/default/streaming_demo_e/checkpoints/', name='checkpoints/', size=0, modificationTime=1795218119136), FileInfo(path='dbfs:/Volumes/main/default/streaming_demo_e/events/', name='events/', size=0, modificationTime=1795218119136)]


In [0]:
import json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

# Ajuste para um Catalog/Schema/Volume válido no seu workspace
base_path = "/Volumes/main/default/streaming_demo_e"
input_path = f"{base_path}/events"
checkpoint_root = f"{base_path}/checkpoints"

dbutils.fs.mkdirs(input_path)
dbutils.fs.mkdirs(checkpoint_root)

arquivos = {
    "events_01.json": [
        {"event_id": "EVT-1", "amount": 100.0, "event_time": "2026-11-20T21:00:00Z"},
        {"event_id": "EVT-2", "amount": 250.0, "event_time": "2026-11-20T21:00:05Z"}
    ],
    "events_02.json": [
        {"event_id": "EVT-3", "amount": 75.0,  "event_time": "2026-11-20T21:00:10Z"},
        {"event_id": "EVT-4", "amount": 400.0, "event_time": "2026-11-20T21:00:15Z"}
    ],
    "events_03.json": [
        {"event_id": "EVT-5", "amount": 190.0, "event_time": "2026-11-20T21:00:20Z"}
    ]
}

for nome_arquivo, registros in arquivos.items():
    conteudo = "\n".join(json.dumps(r) for r in registros)
    dbutils.fs.put(f"{input_path}/{nome_arquivo}", conteudo, overwrite=True)

print("3 arquivos JSON criados com sucesso.")

Wrote 153 bytes.
Wrote 152 bytes.
Wrote 76 bytes.
3 arquivos JSON criados com sucesso.


### Célula 2: Função auxiliar para testar cada Trigger

Define o schema e uma função reutilizável que executa o stream com o trigger escolhido e exibe os micro-batches processados.

In [0]:
import time

event_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("event_time", TimestampType(), True)
])

def criar_stream_reader():
    return (
        spark.readStream
        .format("json")
        .schema(event_schema)
        .option("maxFilesPerTrigger", 1)  # 1 arquivo por micro-batch para facilitar a visualização
        .load(input_path)
    )

def inspecionar_batches(query, label):
    print(f"=== {label} ===")
    print(f"Ativa: {query.isActive} | Total de batches registrados: {len(query.recentProgress)}")
    for p in query.recentProgress:
        print(
            f"  - Batch {p['batchId']} | "
            f"Input rows: {p['numInputRows']} | "
            f"Trigger interval: {p.get('triggerExecutionTimeMs', 'N/A')} ms"
        )

### Célula 3: Trigger 1 — availableNow=True (drena tudo disponível e encerra)

Processa todos os arquivos já existentes no diretório em múltiplos micro-batches (por causa do maxFilesPerTrigger=1) e para automaticamente ao concluir, sem esperar novos arquivos.

In [0]:
ckpt_available = f"{checkpoint_root}/available_now"
table_available = "main.default.events_available_now"

query_available = (
    criar_stream_reader()
    .writeStream
    .format("delta")
    .outputMode("append")
    .queryName("demo_available_now")
    .option("checkpointLocation", ckpt_available)
    .trigger(availableNow=True)
    .toTable(table_available)
)

# Bloqueia até terminar todos os micro-batches disponíveis
query_available.awaitTermination()

inspecionar_batches(query_available, "Trigger: availableNow=True")

=== Trigger: availableNow=True ===
Ativa: False | Total de batches registrados: 3
  - Batch 0 | Input rows: None | Trigger interval: N/A ms
  - Batch 1 | Input rows: None | Trigger interval: N/A ms
  - Batch 2 | Input rows: None | Trigger interval: N/A ms


### Célula 4: Trigger 2 — processingTime (intervalo fixo)


Executa micro-batches em intervalos regulares configurados (ex.: "5 seconds" no demo; em produção costuma ser algo como "1 minute" ou "2 minutes" para controlar custo e uso de recursos). Como fica rodando continuamente até ser interrompida, usamos time.sleep e depois query.stop().

In [0]:
ckpt_interval = f"{checkpoint_root}/processing_time"
table_interval = "main.default.events_processing_time"

query_interval = (
    criar_stream_reader()
    .writeStream
    .format("delta")
    .outputMode("append")
    .queryName("demo_processing_time")
    .option("checkpointLocation", ckpt_interval)
   # .trigger(processingTime="5 seconds")  # Ex.: em produção pode ser "2 minutes"
    .trigger(availableNow=True)  
    .toTable(table_interval)
)

#    .trigger(processingTime="5 seconds")  # Ex.: em produção pode ser "2 minutes"

#[INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED] Trigger type ProcessingTime is not supported for this cluster type.
#Use a different trigger type e.g. AvailableNow, Once. SQLSTATE: 0A000

# Deixa rodar por alguns segundos para observar o comportamento periódico
time.sleep(12)
query_interval.stop()
query_interval.awaitTermination()

inspecionar_batches(query_interval, "Trigger: processingTime='5 seconds'")

=== Trigger: processingTime='5 seconds' ===
Ativa: False | Total de batches registrados: 3
  - Batch 0 | Input rows: None | Trigger interval: N/A ms
  - Batch 1 | Input rows: None | Trigger interval: N/A ms
  - Batch 2 | Input rows: None | Trigger interval: N/A ms


## Célula 4: Trigger 2 — processingTime (intervalo fixo)

Executa micro-batches em intervalos regulares configurados (ex.: "5 seconds" no demo; em produção costuma ser algo como "1 minute" ou "2 minutes" para controlar custo e uso de recursos). Como fica rodando continuamente até ser interrompida, usamos time.sleep e depois query.stop().

In [0]:
ckpt_interval = f"{checkpoint_root}/processing_time"
table_interval = "main.default.events_processing_time"

query_interval = (
    criar_stream_reader()
    .writeStream
    .format("delta")
    .outputMode("append")
    .queryName("demo_processing_time")
    .option("checkpointLocation", ckpt_interval)
    #.trigger(processingTime="5 seconds")  # Ex.: em produção pode ser "2 minutes"
    .trigger(availableNow=True)  
    .toTable(table_interval)
)

# Deixa rodar por alguns segundos para observar o comportamento periódico
time.sleep(12)
query_interval.stop()
query_interval.awaitTermination()

inspecionar_batches(query_interval, "Trigger: processingTime='5 seconds'")

=== Trigger: processingTime='5 seconds' ===
Ativa: False | Total de batches registrados: 1
  - Batch 3 | Input rows: None | Trigger interval: N/A ms


## Célula 5: Trigger 3 — Default (as soon as possible — sem .trigger(...))

Quando nenhum .trigger(...) é informado, o Spark inicia o próximo micro-batch assim que o anterior termina, buscando baixa latência em micro-batch sem intervalo artificial de espera.

In [0]:
ckpt_default = f"{checkpoint_root}/default_trigger"
table_default = "main.default.events_default"

query_default = (
    criar_stream_reader()
    .writeStream
    .format("delta")
    .outputMode("append")
    .queryName("demo_default_asap")
    .option("checkpointLocation", ckpt_default)
    # Sem .trigger(...) -> executa o próximo micro-batch assim que o anterior conclui
    .trigger(availableNow=True)  # tive que colocar para testar
    .toTable(table_default)
)

time.sleep(6)
query_default.stop()
query_default.awaitTermination()

inspecionar_batches(query_default, "Trigger: Default (as soon as possible)")

=== Trigger: Default (as soon as possible) ===
Ativa: False | Total de batches registrados: 1
  - Batch 3 | Input rows: None | Trigger interval: N/A ms


In [0]:
# Opcional: limpar tabelas e diretórios temporários
#for tbl in [table_available, table_interval, table_default]:
#    spark.sql(f"DROP TABLE IF EXISTS {tbl}")

#dbutils.fs.rm(base_path, recurse=True)
#print("Limpeza concluída.")

| Trigger | Quando executa | Encerra sozinho? | Caso de uso típico |
| :--- | :--- | :--- | :--- |
| Default (sem .trigger()) | Assim que o micro-batch anterior termina | Não (roda continuamente) | Baixa latência em micro-batch sem atraso programado |
| processingTime | A cada intervalo fixo configurado | Não (roda continuamente) | Controle de custo/CPU e cadência previsível (ex.: a cada 1–5 min) |
| availableNow=True | Processa tudo que já chegou (em 1 ou mais batches) | Sim (ao terminar o disponível) | Jobs incrementais agendados, backfill e demos em notebook |
| realTime (Spark 4) | Execução contínua voltada a latência sub-segundo | Não (roda continuamente) | Casos operacionais de altíssima urgência (ex.: antifraude em tempo real) |

A opção .option("maxFilesPerTrigger", 1) no Apache Spark Structured Streaming (usado no Databricks) define no máximo quantos arquivos novos serão lidos e processados em cada micro-batch (micro-lote) quando você usa uma fonte baseada em arquivos (como cloudFiles / Auto Loader, CSV, JSON, Parquet, etc.).

No seu comentário:

# 1 arquivo por micro-batch para facilitar a visualização, para salvar no databricks, por spark

Isso significa que, se chegarem 10 arquivos na pasta de origem ao mesmo tempo, o Spark não vai ler os 10 de uma vez em um único lote. Em vez disso, ele vai processar:

Micro-batch 1: 1 arquivo → salva no destino (ex.: tabela Delta no Databricks)

Micro-batch 2: 1 arquivo → salva no destino

Micro-batch 3: 1 arquivo → salva no destino

...e assim por diante, até alcançar os arquivos novos.

Por que isso ajuda na visualização e nos testes?
Acompanhamento passo a passo: Você consegue observar cada ciclo do stream (batch 0, batch 1, batch 2...) e ver os dados entrando na tabela aos poucos, sem misturar um volume grande de uma só vez.

Controle de carga e memória: Em ambientes de teste ou com arquivos pesados, processar 1 arquivo por vez evita picos de uso de memória/shuffle no cluster.

Simulação de chegada gradual: Mesmo que você jogue vários arquivos na pasta de uma vez, o comportamento se aproxima de arquivos chegando de forma incremental.

In [0]:
#df_stream = (
#    spark.readStream
#    .format("cloudFiles")  # ou "json", "csv", "parquet"
#    .option("cloudFiles.format", "json")
#    .option("maxFilesPerTrigger", 1)  # Lê no máximo 1 arquivo por micro-batch
#    .load("/Volumes/catalog/schema/raw_volume/input/")
#)

#query = (
#    df_stream.writeStream
#    .format("delta")
#    .option("checkpointLocation", "/Volumes/catalog/schema/raw_volume/checkpoints/minha_tabela/")
#    .toTable("catalog.schema.minha_tabela")
#)


#### Output Modes

Output modes define how Structured Streaming writes results to sinks:

    • append (default)

        ◦ Only adds new records to the sink
        ◦ Best for simple (or stateless) queries without aggregations
        
    • update (ideal for dashboards and real-time metrics)

        ◦ Modifies existing records and adds new ones
        ◦ Only outputs records that changed since the last trigger

    • complete (ideal for running totals or leaderboards)

        ◦ Writes the entire result table to the sink each time
        ◦ Required for some aggregations
ℹ️

# Checkpointing and Fault Tolerance

The checkpoint location is the durable memory of a streaming query:

    • Records which input has been processed (source offsets) and any operator state
    • Enables recovery: restart the query with the same checkpoint and it resumes exactly where it left off
    • Enables exactly-once end-to-end guarantees when combined with replayable sources (files, Kafka, Delta)     
    and idempotent sinks (Delta)
    • One checkpoint location per query - never share one between queries


In [0]:
#query = (
#    df.writeStream
#    .option("checkpointLocation", "/Volumes/catalog/schema/checkpoints/orders_bronze")
#    .toTable("orders_bronze")
#)


### Monitoring Streaming Queries

Understanding your streaming application's health:

    • Spark UI: the Structured Streaming tab shows active queries with per-batch progress details
    • External monitoring tools: Datadog, Grafana, 
    Prometheus, and similar can track processing rates, latency, memory usage, and garbage collection
    • Inspection from code:

In [0]:
# Check if a DataFrame is streaming
#df.isStreaming            # returns True or False


# Get the unique identifier for a query
#query.id


# Get the current state of a query
#query.status


# Inspect the most recent progress report (rates, durations, offsets)
#query.lastProgress


# List all active queries on the session
#spark.streams.active

### Célula 1: Preparar diretórios isolados e arquivos iniciais

Cria caminhos separados para cada query (seguindo a regra de um checkpoint por query) e escreve o primeiro lote de dados.

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS main.default.streaming_demo_gh")
base_path = "/Volumes/main/default/streaming_demo_gh"
dbutils.fs.mkdirs(f"{base_path}/orders")
dbutils.fs.mkdirs(f"{base_path}/checkpoints/orders_bronze")
dbutils.fs.mkdirs(f"{base_path}/checkpoints/high_value_orders")

print("Volume e diretórios criados:", dbutils.fs.ls(base_path))

Volume e diretórios criados: [FileInfo(path='dbfs:/Volumes/main/default/streaming_demo_gh/checkpoints/', name='checkpoints/', size=0, modificationTime=1795220808244), FileInfo(path='dbfs:/Volumes/main/default/streaming_demo_gh/orders/', name='orders/', size=0, modificationTime=1795220808244)]


In [0]:
import json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

# Ajuste para um Catalog/Schema/Volume válido no seu workspace
base_path = "/Volumes/main/default/streaming_demo_gh"
input_path = f"{base_path}/orders"

# Regra de ouro: um checkpointLocation dedicado por query (nunca compartilhar)
checkpoint_orders_bronze = f"{base_path}/checkpoints/orders_bronze"
checkpoint_high_value = f"{base_path}/checkpoints/high_value_orders"

table_bronze = "main.default.orders_bronze"
table_high_value = "main.default.high_value_orders"

dbutils.fs.mkdirs(input_path)
dbutils.fs.mkdirs(checkpoint_orders_bronze)
dbutils.fs.mkdirs(checkpoint_high_value)

batch_1 = [
    {"order_id": "ORD-301", "amount": 120.00, "event_time": "2026-11-20T22:00:00Z"},
    {"order_id": "ORD-302", "amount": 850.00, "event_time": "2026-11-20T22:00:05Z"}
]

dbutils.fs.put(f"{input_path}/orders_01.json", "\n".join(json.dumps(r) for r in batch_1), overwrite=True)
print("Lote 1 (orders_01.json) criado com sucesso.")

Wrote 157 bytes.
Lote 1 (orders_01.json) criado com sucesso.


## Célula 2: Iniciar query com Checkpoint e inspecionar estado/monitoramento (isStreaming, id, status, lastProgress, active)

Verifica se o DataFrame é de streaming, inicia a query apontando para o checkpoint dedicado e consulta os atributos de monitoramento via código.

In [0]:
from pyspark.sql.functions import col

order_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("event_time", TimestampType(), True)
])

orders_stream_df = (
    spark.readStream
    .format("json")
    .schema(order_schema)
    .option("maxFilesPerTrigger", 1)
    .load(input_path)
)

# Confirma em código se o DataFrame é streaming
print("orders_stream_df.isStreaming:", orders_stream_df.isStreaming)

orders_stream_df.isStreaming: True


In [0]:


# Inicia a query com checkpointLocation dedicado
query_bronze = (
    orders_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .queryName("orders_bronze_query")
    .option("checkpointLocation", checkpoint_orders_bronze)
    .trigger(availableNow=True)
    .toTable(table_bronze)
)

# Inspeção programática da query
print("Query ID:", query_bronze.id)
print("Query Status:", query_bronze.status)
print("Queries ativas na sessão:", [q.name for q in spark.streams.active])

query_bronze.awaitTermination()

print("Last Progress (resumo de taxas e durações):")
if query_bronze.lastProgress:
    print("  - Batch ID:", query_bronze.lastProgress.get("batchId"))
    print("  - Input rows/s:", query_bronze.lastProgress.get("inputRowsPerSecond"))
    print("  - Processed rows/s:", query_bronze.lastProgress.get("processedRowsPerSecond"))
    print("  - Duração total (ms):", query_bronze.lastProgress.get("durationMs"))

Query ID: a528aa52-cb1b-4af0-97ec-c332b15bf293
Query Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}
Queries ativas na sessão: ['orders_bronze_query']
Last Progress (resumo de taxas e durações):
  - Batch ID: 0
  - Input rows/s: None
  - Processed rows/s: None
  - Duração total (ms): {'triggerExecution': 1719, 'queryPlanning': 141, 'collectSourceMetrics': 0, 'getBatch': 182, 'commitOffsets': 82, 'addBatch': 915, 'latestOffset': 82, 'commitBatch': 192, 'walCommit': 86}


### Célula 3: Simular interrupção, chegada de novos dados e recuperação via Checkpoint (exactly-once)

Adiciona um novo arquivo (orders_02.json) e reinicia a query usando o mesmo checkpointLocation. O Spark lê apenas os novos offsets não processados e grava na tabela Delta sem duplicar ORD-301 e ORD-302.

In [0]:
batch_2 = [
    {"order_id": "ORD-303", "amount": 45.00,  "event_time": "2026-11-20T22:01:00Z"},
    {"order_id": "ORD-304", "amount": 1200.00,"event_time": "2026-11-20T22:01:10Z"}
]

dbutils.fs.put(f"{input_path}/orders_02.json", "\n".join(json.dumps(r) for r in batch_2), overwrite=True)

# Reinicia a query apontando para o mesmo checkpointLocation
query_bronze_recovered = (
    orders_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .queryName("orders_bronze_query")
    .option("checkpointLocation", checkpoint_orders_bronze)
    .trigger(availableNow=True)
    .toTable(table_bronze)
)

query_bronze_recovered.awaitTermination()

# Confere que os 4 pedidos estão presentes sem duplicação
display(spark.table(table_bronze).orderBy("order_id"))

Wrote 157 bytes.


order_id,amount,event_time
ORD-301,120.0,2026-11-20T22:00:00.000Z
ORD-302,850.0,2026-11-20T22:00:05.000Z
ORD-303,45.0,2026-11-20T22:01:00.000Z
ORD-304,1200.0,2026-11-20T22:01:10.000Z


## Célula 4: Segunda query com seu próprio Checkpoint (nunca reutilize o mesmo checkpoint)

Demonstra uma segunda query derivada do mesmo stream gravando em outra tabela, exigindo obrigatoriamente um checkpointLocation distinto para não corromper os offsets ou o estado.

In [0]:
high_value_df = orders_stream_df.filter(col("amount") >= 500.0)

query_high_value = (
    high_value_df.writeStream
    .format("delta")
    .outputMode("append")
    .queryName("high_value_orders_query")
    .option("checkpointLocation", checkpoint_high_value)  # Checkpoint próprio desta query
    .trigger(availableNow=True)
    .toTable(table_high_value)
)

query_high_value.awaitTermination()
display(spark.table(table_high_value).orderBy("order_id"))

order_id,amount,event_time
ORD-302,850.0,2026-11-20T22:00:05.000Z
ORD-304,1200.0,2026-11-20T22:01:10.000Z


In [0]:
high_value_df

DataFrame[order_id: string, amount: double, event_time: timestamp]

In [0]:
display(high_value_df, checkpointLocation=f"{base_path}/checkpoints/display_high_value")  # display() de DF de streaming exige checkpointLocation explícito, dedicado a esta query

Checkpointing to /Volumes/main/default/streaming_demo_gh/checkpoints/display_high_value


### Célula 5: Limpeza (opcional)

Remove as tabelas de exemplo e os diretórios temporários criados no Volume.

In [0]:
 
for tbl in [table_bronze, table_high_value]:
   display(spark.sql(f"SELECT * from {tbl}"))

order_id,amount,event_time
ORD-303,45.0,2026-11-20T22:01:00.000Z
ORD-304,1200.0,2026-11-20T22:01:10.000Z
ORD-301,120.0,2026-11-20T22:00:00.000Z
ORD-302,850.0,2026-11-20T22:00:05.000Z


order_id,amount,event_time
ORD-304,1200.0,2026-11-20T22:01:10.000Z
ORD-302,850.0,2026-11-20T22:00:05.000Z


In [0]:
for tbl in [table_bronze, table_high_value]:
    spark.sql(f"DROP TABLE IF EXISTS {tbl}")

dbutils.fs.rm(base_path, recurse=True)
print("Ambiente de teste limpo.")

Ambiente de teste limpo.


O checkpoint (checkpointLocation) funciona como a memória persistente e auditável de uma consulta de Structured Streaming. Em vez de depender apenas da memória RAM do cluster ou da execução contínua do processo, o Spark grava no diretório de checkpoint os metadados necessários para saber o que já foi lido, o que já foi confirmado e qual é o estado atual da lógica.

O que exatamente ele salva dentro da pasta?
Quando você roda uma query de streaming com .option("checkpointLocation", ...), o Spark cria subpastas com informações essenciais:

Offsets (offsets/): Registra até qual ponto a fonte de dados foi lida em cada micro-batch (por exemplo, quais arquivos JSON já foram consumidos, quais offsets/partições do Kafka já foram lidos ou qual versão do Delta já entrou no batch).

Commits (commits/): Confirma quais micro-batches foram concluídos e gravados com sucesso no sink de destino. Se uma falha acontecer no meio de um batch, o Spark compara offsets e commits ao reiniciar para saber exatamente qual batch precisa ser refeito ou retomado.

Estado dos operadores (state/): Quando a query tem operações com estado (como agregações por janela, groupBy, deduplicação ou stream-stream joins), salva os snapshots/arquivos de estado (frequentemente gerenciados com RocksDB ou estado em memória/disco) para que o cálculo continue de onde parou.

Metadados da query (metadata/ e arquivos de configuração): Guarda o ID da query e validações de compatibilidade (como esquema/configuração) para evitar que um checkpoint seja reaproveitado indevidamente por outra consulta incompatível.